# NB01 — Exploration, nettoyage et statistiques des séquences VHH
**Projet 1.1 — Phase A, tâches 1.2 à 1.6**

## Rôle de ce notebook
Ce notebook est le **seul endroit où les séquences sont nettoyées** (gaps, alphabet, longueur, doublons exacts). Il étudie ensuite le jeu obtenu.

| | Chemin |
|---|---|
| Entrée | `data/processed/slim/*.csv` (produit par `scripts/slim_columns.py` : seulement les colonnes utiles, lignes inchangées) |
| Sorties | `data/processed/<DATA_VERSION>/vhh_clean.csv`, `cleaning_report.csv`, `manifest.json` |
| Figures | `results/NB01/<DATA_VERSION>/` |

`DATA_VERSION` vaut **`toy`** en local (fausses données) et **`v1`** sur Colab (données réelles). Elle est choisie automatiquement par `utils.get_data_version()` : on ne la modifie pas à la main.

## Formules (exactes, sans epsilon)
- Fréquence : f(a) = count(a) / N_total
- Entropie par position : H(i) = −Σₐ pᵢ(a) log₂ pᵢ(a), avec la convention 0·log 0 = 0 (masque exact, pas d'epsilon)
- PWM : PWM(i, a) = log₂( pᵢ(a) / p_bg(a) ) ; vaut −∞ si pᵢ(a) = 0 (affiché en gris)
- Information : I(i) = log₂(20) − H(i) (Schneider & Stephens, 1990)
- Redondance exacte : (N − N_uniques) / N

## Références
Olsen et al. 2022 (OAS) · Schneider & Stephens 1990 (logos) · Fu et al. 2012 (CD-HIT)


run these : before NB01 -> NB02 
1) python scripts/make_toy_data.py
2) python scripts/slim_columns.py --inputs data/toy/raw/toy_oas_raw.csv --out-dir data/toy/processed/slim

In [ ]:
# ==============================================================================
# Cellule 1 : Bootstrap (local ET Google Colab), chemins, version des données
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "main"  # branche clonée sur Colab (doit contenir ce notebook et scripts/)
        
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _find_root(start):
    """Remonte depuis `start` jusqu'au dossier qui contient utils.py."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


ROOT_DIR = _find_root(Path.cwd())

if ROOT_DIR is None and IN_COLAB:           # Colab
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)], check=True)
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError("utils.py introuvable : ouvrez le notebook dans le dépôt Recherche_AI.")

# Colab uniquement : monter le Drive AVANT get_paths() ; AI_PROJECT_DIR active le mode « données réelles »
if IN_COLAB:                           # Colab: mount Drive
    from google.colab import drive
    drive.mount("/content/drive")
    _drive = Path("/content/drive/MyDrive/AI_Project_Shared")
    if not _drive.exists():
        raise FileNotFoundError(f"Dossier Drive introuvable : {_drive}. Ajoutez le raccourci dans Mon Drive.")
    os.environ.setdefault("AI_PROJECT_DIR", str(_drive))

sys.path.insert(0, str(ROOT_DIR))

import json, hashlib
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (get_paths, get_data_version, set_seed, STANDARD_AA, MIN_LEN, MAX_LEN,
                   compute_aa_composition)

SEED = 42
set_seed(SEED)

PATHS = get_paths()
DATA_VERSION = get_data_version()
SLIM_DIR = PATHS["processed"] / "slim"
OUT_DIR = PATHS["processed"] / DATA_VERSION
FIG_DIR = PATHS["results"] / "NB01" / DATA_VERSION
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODE          : {'DONNÉES RÉELLES' if DATA_VERSION != 'toy' else 'TOY (fausses données)'}")
print(f"DATA_VERSION  : {DATA_VERSION}")
print(f"Entrée (slim) : {SLIM_DIR}")
print(f"Sortie        : {OUT_DIR}")
print(f"Seed          : {SEED}")


In [ ]:
# ==============================================================================
# Cellule 2 : Chargement des fichiers slim (Tâche 1.1)
# ==============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


slim_files = sorted(SLIM_DIR.glob("*.csv"))
if not slim_files:
    raise FileNotFoundError(
        f"Aucun fichier .csv dans {SLIM_DIR}.\n"
        f"Lancez d'abord : python scripts/slim_columns.py --inputs <fichiers raw> --out-dir {SLIM_DIR}")

frames, metadata = [], {}
for p in slim_files:
    with open(p, encoding="utf-8") as f:
        first = f.readline().rstrip("\n")
    has_meta = "sequence_alignment_aa" not in first.split(",")      # 1re ligne = métadonnées JSON d'OAS
    metadata[p.name] = json.loads(first) if has_meta else {}
    d = pd.read_csv(p, skiprows=1 if has_meta else 0, usecols=["sequence_alignment_aa"],
                    dtype=str, keep_default_na=False)
    d = d.rename(columns={"sequence_alignment_aa": "aligned"})
    d["source_file"] = p.name
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(f"{len(slim_files)} fichier(s) slim, {len(df):,} lignes au total")
meta_table = pd.DataFrame(metadata).T
meta_table.insert(0, "lignes", [int((df["source_file"] == k).sum()) for k in meta_table.index])
display(meta_table)


In [ ]:
# ==============================================================================
# Cellule 3 : Nettoyage exact — gaps, vides, alphabet (Tâche 1.2)
# ==============================================================================
ALPHABET_PATTERN = "[" + STANDARD_AA + "]+"        # équivalent de ^[ACDEFGHIKLMNPQRSTVWY]+$ avec fullmatch
report = []


def log(step, n, reason=""):
    before = report[-1]["rows_after"] if report else n
    report.append({"step": step, "rows_before": before, "rows_after": n,
                   "removed": before - n, "reason": reason})


log("1. lignes brutes", len(df), "fichiers slim")

df["seq"] = df["aligned"].str.strip().str.upper()
n_gapped = int(df["seq"].str.contains("-", regex=False).sum())
df["seq"] = df["seq"].str.replace("-", "", regex=False)
df = df[df["seq"] != ""]
log("2. gaps retirés, séquences vides supprimées", len(df),
    f"{n_gapped:,} séquences contenaient des gaps '-' (gaps retirés, séquences gardées) ; rejet = séquence vide")

ok = df["seq"].str.fullmatch(ALPHABET_PATTERN)
bad_letters = Counter(df.loc[~ok, "seq"].str.extract(f"([^{STANDARD_AA}])")[0])
df = df[ok].copy()
reason = ", ".join(f"{k}: {v:,}" for k, v in bad_letters.most_common(6)) or "aucune"
log("3. alphabet des 20 acides aminés", len(df), f"1re lettre hors alphabet — {reason}")

assert df["seq"].str.fullmatch(ALPHABET_PATTERN).all()
display(pd.DataFrame(report))


In [ ]:
# ==============================================================================
# Cellule 4 : Distribution des longueurs AVANT le filtre de longueur (Tâche 1.3)
# ==============================================================================
df["len"] = df["seq"].str.len()
L = df["len"]
mode_len = int(L.mode().iloc[0])
n_short, n_long = int((L < MIN_LEN).sum()), int((L > MAX_LEN).sum())

display(L.describe(percentiles=[.25, .5, .75]).to_frame("longueur").T.round(2))
print(f"Longueur modale : {mode_len}")
print(f"Bornes actuelles (utils.py) : MIN_LEN={MIN_LEN}, MAX_LEN={MAX_LEN}")
print(f"  trop courtes (< {MIN_LEN}) : {n_short:,}    trop longues (> {MAX_LEN}) : {n_long:,}")
display(df.groupby("source_file")["len"].describe()[["count", "min", "50%", "max"]])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(L, bins=np.arange(L.min(), L.max() + 2) - 0.5, color="steelblue")
ax.axvline(MIN_LEN, color="red", ls="--", label=f"MIN_LEN = {MIN_LEN}")
ax.axvline(MAX_LEN, color="orange", ls="--", label=f"MAX_LEN = {MAX_LEN}")
ax.set_xlabel("Longueur (résidus)"); ax.set_ylabel("Nombre de séquences")
ax.set_title("Distribution des longueurs (avant filtre de longueur)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "lengths_histogram.png", dpi=150); plt.show()


**Décision sur les bornes.** Les bornes `MIN_LEN` / `MAX_LEN` (fichier `utils.py`) sont *provisoires*. Regardez l'histogramme ci-dessus sur les données réelles : si le pic principal est coupé ou si une sous-population importante est exclue, changez les valeurs dans `utils.py` (une seule fois, pour tous les notebooks), relancez ce notebook et notez la raison dans le rapport. Les bornes utilisées sont écrites dans `manifest.json` et relues par NB02.

In [ ]:
# ==============================================================================
# Cellule 5 : Filtre de longueur et déduplication exacte (Tâches 1.2 et 1.5)
# ==============================================================================
df = df[df["len"].between(MIN_LEN, MAX_LEN)]
log(f"4. longueur entre {MIN_LEN} et {MAX_LEN}", len(df), f"trop courtes: {n_short:,} ; trop longues: {n_long:,}")

n_before_dedup = len(df)
dup_counts = df["seq"].value_counts()
n_unique = len(dup_counts)
redundancy = (n_before_dedup - n_unique) / n_before_dedup
print(f"Séquences uniques : {n_unique:,} / {n_before_dedup:,}  ->  taux de redondance exacte = {redundancy:.2%}")
print(f"Séquences présentes plusieurs fois : {(dup_counts > 1).sum():,} ; occurrences max : {dup_counts.iloc[0]:,}")

clean = df.drop_duplicates(subset="seq", keep="first").reset_index(drop=True)
log("5. doublons exacts supprimés", len(clean), f"taux de redondance = {redundancy:.2%}")

if len(clean) == 0:
    raise ValueError("Aucune séquence après nettoyage : vérifiez les bornes de longueur et les fichiers d'entrée.")

clean[["seq"]].rename(columns={"seq": "sequence"}).to_csv(OUT_DIR / "vhh_clean.csv", index=False)
pd.DataFrame(report).to_csv(OUT_DIR / "cleaning_report.csv", index=False)
display(pd.DataFrame(report))
print("Séquences gardées par fichier :")
display(clean["source_file"].value_counts().to_frame("séquences"))
print(f"Écrit : {OUT_DIR / 'vhh_clean.csv'}")


### Pourquoi la déduplication exacte ne suffit pas (à reformuler avec vos mots pour le rapport)
`set()` n'élimine que les séquences identiques à 100 %. Dans un répertoire d'anticorps, l'hypermutation somatique produit des **quasi-doublons** (un ou deux résidus de différence) qui restent tous les deux. Avec un split aléatoire, un quasi-doublon peut se trouver dans *train* et l'autre dans *test* : le modèle a déjà vu presque la séquence de test, donc la vraisemblance test est trop optimiste (fuite d'information). La solution prévue en Phase 3 est un clustering à 90 % d'identité (CD-HIT, Fu et al. 2012, ou MMseqs2) puis un split par clusters.

In [ ]:
# ==============================================================================
# Cellule 6 : Composition globale en acides aminés (Tâche 1.3)
# ==============================================================================
def composition(seqs: pd.Series) -> np.ndarray:
    """f(a) = count(a) / N_total, dans l'ordre de STANDARD_AA."""
    counts = np.array([int(seqs.str.count(a).sum()) for a in STANDARD_AA], dtype=np.int64)
    return counts / counts.sum()


p_bg = composition(clean["seq"])
display(pd.DataFrame({"f(a)": p_bg}, index=list(STANDARD_AA)).T.round(4))
print(f"Somme des fréquences : {p_bg.sum():.12f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(list(STANDARD_AA), p_bg, color="steelblue")
ax.set_ylabel("Fréquence"); ax.set_title("Composition globale en acides aminés")
fig.tight_layout(); fig.savefig(FIG_DIR / "composition.png", dpi=150); plt.show()


### PWM et information par position (Tâche 1.4) — méthode et limite
Une PWM suppose des séquences **alignées**. Les gaps ayant été retirés, on utilise ici la **fenêtre alignable la plus simple** : toutes les séquences de la longueur modale L\*, comparées position par position. C'est une approximation : deux séquences de même longueur ne sont pas forcément alignées si leurs CDR diffèrent (indels compensés), donc la conservation des cadres (FR) est lisible mais celle des CDR est floue. Un vrai alignement IMGT (Phase 2) corrigera cela. Autre biais : avec peu de séquences, H(i) est sous-estimée (donc I(i) surestimée).

In [ ]:
# ==============================================================================
# Cellule 7 : PWM, entropie de Shannon et information par position (Tâche 1.4)
# ==============================================================================
L_STAR = int(clean["len"].mode().iloc[0])
window = clean.loc[clean["len"] == L_STAR, "seq"]
M = len(window)
print(f"Fenêtre alignable : longueur modale L* = {L_STAR}, {M:,} séquences")
if M < 30:
    print("ATTENTION : moins de 30 séquences dans la fenêtre : PWM peu fiable (normal sur les données toy).")

arr = np.frombuffer("".join(window).encode("ascii"), dtype=np.uint8).reshape(M, L_STAR)
counts = np.stack([(arr == ord(a)).sum(axis=0) for a in STANDARD_AA], axis=1)      # (L*, 20)
p = counts / M                                                                       # lignes de somme 1

p_safe = np.where(p > 0, p, 1.0)                           # masque exact (pas d'epsilon) : 0·log 0 = 0
pwm = np.where(p > 0, np.log2(p_safe / p_bg[None, :]), -np.inf)
H = -np.sum(np.where(p > 0, p * np.log2(p_safe), 0.0), axis=1)
I = np.log2(20) - H

pwm_masked = np.ma.masked_invalid(np.where(np.isfinite(pwm), pwm, np.nan))
cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("lightgrey")
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pwm_masked.T, aspect="auto", cmap=cmap, vmin=-3, vmax=3)
ax.set_yticks(range(20)); ax.set_yticklabels(list(STANDARD_AA))
ax.set_xlabel("Position (0 = premier résidu)"); ax.set_title("PWM : log2( p_i(a) / p_bg(a) ) — gris : p_i(a) = 0")
fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(FIG_DIR / "pwm_heatmap.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(1, L_STAR + 1), I, color="steelblue")
ax.set_ylim(0, np.log2(20)); ax.set_xlabel("Position (1 = premier résidu)"); ax.set_ylabel("Information (bits)")
ax.set_title(f"Information par position I(i) = log2(20) − H(i)   (L* = {L_STAR}, {M:,} séquences)")
fig.tight_layout(); fig.savefig(FIG_DIR / "information_curve.png", dpi=150); plt.show()


In [ ]:
# ==============================================================================
# Cellule 8 : Validation automatique et manifest (reproductibilité)
# ==============================================================================
sample = clean["seq"].iloc[:20000]
ref = compute_aa_composition(sample.tolist())
reread = pd.read_csv(OUT_DIR / "vhh_clean.csv", dtype=str, keep_default_na=False)

checks = {
    "100 % des séquences passent la regex des 20 AA": bool(clean["seq"].str.fullmatch(ALPHABET_PATTERN).all()),
    "toutes les longueurs sont dans les bornes": bool(clean["len"].between(MIN_LEN, MAX_LEN).all()),
    "aucun doublon exact": bool(clean["seq"].is_unique),
    "fréquences f(a) : somme = 1": bool(np.isclose(p_bg.sum(), 1.0)),
    "PWM : chaque position somme à 1": bool(np.allclose(p.sum(axis=1), 1.0)),
    "composition = utils.compute_aa_composition (échantillon)":
        bool(np.allclose([ref[a] for a in STANDARD_AA], composition(sample))),
    "vhh_clean.csv relu = séquences en mémoire": reread["sequence"].tolist() == clean["seq"].tolist(),
}
for name, ok_ in checks.items():
    print(("OK   " if ok_ else "ÉCHEC"), name)
assert all(checks.values()), "Au moins une validation a échoué"


def git_commit(repo):
    try:
        return subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unknown"


manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "NB01",
    "git_commit": git_commit(ROOT_DIR),
    "data_version": DATA_VERSION,
    "parameters": {"min_len": MIN_LEN, "max_len": MAX_LEN, "seed": SEED, "alphabet": STANDARD_AA,
                   "pwm_window_length": L_STAR, "pwm_window_sequences": int(M)},
    "inputs": [{"file": p_.name, "sha256": sha256_of(p_), "metadata": metadata[p_.name]} for p_ in slim_files],
    "steps": report,
    "outputs": {f: {"rows": int(len(reread)) if f == "vhh_clean.csv" else int(len(report)),
                    "sha256": sha256_of(OUT_DIR / f)} for f in ("vhh_clean.csv", "cleaning_report.csv")},
}
with open(OUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f"\nManifest écrit : {OUT_DIR / 'manifest.json'}")


In [ ]:
# ==============================================================================
# Cellule 9 : Fiche de synthèse — les chiffres à recopier dans le rapport (Tâche 1.6)
# ==============================================================================
order = np.argsort(p_bg)[::-1]
top_pos = np.argsort(I)[::-1][:5]
low_pos = np.argsort(I)[:5]
fiche = {
    "version des données": DATA_VERSION,
    "lignes brutes": f"{report[0]['rows_before']:,}",
    "séquences propres et uniques": f"{len(clean):,}",
    "rejet total": f"{1 - len(clean) / report[0]['rows_before']:.2%}",
    "longueur min / médiane / modale / max": f"{clean['len'].min()} / {int(clean['len'].median())} / {L_STAR} / {clean['len'].max()}",
    "redondance exacte": f"{redundancy:.2%}",
    "3 AA les plus fréquents": ", ".join(f"{STANDARD_AA[i]} ({p_bg[i]:.1%})" for i in order[:3]),
    "3 AA les plus rares": ", ".join(f"{STANDARD_AA[i]} ({p_bg[i]:.1%})" for i in order[-3:]),
    "5 positions les plus informatives (1-indexées)": ", ".join(f"{i + 1} ({I[i]:.2f} bits)" for i in top_pos),
    "5 positions les moins informatives": ", ".join(f"{i + 1} ({I[i]:.2f} bits)" for i in low_pos),
}
for k, v in fiche.items():
    print(f"{k:<48} {v}")


## Analyse rédigée — Tâche 1.6 (demi-page du rapport)
*À rédiger vous-mêmes à partir de la fiche de synthèse ci-dessus (données réelles, pas toy).*

1. **Longueurs** : la distribution est-elle unimodale ? Y a-t-il des sous-populations ? Les bornes choisies sont-elles justifiées par l'histogramme ?
2. **Conservation** : quelles positions ont une information élevée ? Retrouve-t-on des cadres conservés (FR) et des zones variables (CDR) ? Quelles sont les limites de la fenêtre de longueur fixe ?
3. **Redondance** : quel est le taux de doublons exacts ? Pourquoi la déduplication à identité partielle sera-t-elle nécessaire en Phase 3 ?

> Rappel : ne jamais recopier ici les chiffres d'un run **toy**.
